# 05 RandomForestClassifier Model Selection

In diesem Notebook wird der RandomForestClassifier mit 5-fold Group Cross-Validation nach `subject` bewertet. Anschließend wird das beste Random-Forest-Modell auf dem festgelegten Testset mit den Subjects 27 bis 30 ausgewertet.

In [ ]:
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

PROCESSED_DIR = Path("../../data/processed")
RESULTS_DIR = Path("../../results")
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Daten laden

In [ ]:
train_data = pd.read_csv(PROCESSED_DIR / "train_data.csv")
test_data = pd.read_csv(PROCESSED_DIR / "test_data.csv")

with open(PROCESSED_DIR / "feature_columns.txt", "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f]

print("Train data:", train_data.shape)
print("Test data:", test_data.shape)
print("Anzahl Features:", len(feature_cols))

## Features, Zielvariable und Groups definieren

In [ ]:
target_col = "activity"
group_col = "subject"

X_train = train_data[feature_cols]
y_train = train_data[target_col]
groups_train = train_data[group_col]

X_test = test_data[feature_cols]
y_test = test_data[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("groups_train:", groups_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

## Random-Forest-Modell definieren

Random Forest ist ein Ensemble-Verfahren aus vielen Entscheidungsbäumen. Im Gegensatz zu kNN oder SVM benötigt Random Forest normalerweise keine Skalierung der Features.

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_model

## Cross-Validation und Hyperparameter-Suche

Die Modellwahl erfolgt mit 5-fold Group Cross-Validation. Die Gruppierung erfolgt nach `subject`, damit Messungen derselben Person nicht gleichzeitig in Trainings- und Validierungsdaten vorkommen.

In [ ]:
cv = GroupKFold(n_splits=5)

rf_param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True],
}

In [ ]:
rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_distributions,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

rf_random_search.fit(
    X_train,
    y_train,
    groups=groups_train,
)

## Beste Parameter und Cross-Validation-Ergebnis

In [ ]:
print("Beste Parameter:")
print(rf_random_search.best_params_)

print("\nBeste CV Accuracy:")
print(rf_random_search.best_score_)

print("\nBester CV Error:")
print(1 - rf_random_search.best_score_)

In [ ]:
rf_search_results = pd.DataFrame(rf_random_search.cv_results_)

selected_columns = [
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "std_train_score",
    "mean_fit_time",
    "mean_score_time",
    "params",
]

rf_search_results_table = rf_search_results[selected_columns].sort_values(
    by="mean_test_score",
    ascending=False,
)

rf_search_results_table.head(10)

In [ ]:
rf_search_results_table.to_csv(
    TABLE_DIR / "rf_random_search_results.csv",
    index=False,
)

print("Gespeichert:", TABLE_DIR / "rf_random_search_results.csv")

## Bestes Random-Forest-Modell erneut mit Cross-Validation auswerten

In [ ]:
best_rf_model = rf_random_search.best_estimator_

rf_cv_results = cross_validate(
    best_rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

rf_summary = pd.DataFrame({
    "model": ["RandomForestClassifier_tuned"],
    "mean_cv_accuracy": [rf_cv_results["test_score"].mean()],
    "std_cv_accuracy": [rf_cv_results["test_score"].std()],
    "mean_cv_error": [1 - rf_cv_results["test_score"].mean()],
    "mean_train_accuracy": [rf_cv_results["train_score"].mean()],
    "mean_fit_time": [rf_cv_results["fit_time"].mean()],
    "mean_score_time": [rf_cv_results["score_time"].mean()],
})

rf_summary

In [ ]:
rf_summary.to_csv(
    TABLE_DIR / "rf_tuned_cv_summary.csv",
    index=False,
)

print("Gespeichert:", TABLE_DIR / "rf_tuned_cv_summary.csv")

## Bewertung auf dem Testset

Nach der Modellwahl wird das beste Random-Forest-Modell auf den Trainingsdaten trainiert und anschließend auf dem festgelegten Testset mit den Subjects 27 bis 30 bewertet.

In [ ]:
best_rf_model.fit(X_train, y_train)

y_pred_rf = best_rf_model.predict(X_test)

rf_test_accuracy = accuracy_score(y_test, y_pred_rf)
rf_test_error = 1 - rf_test_accuracy

print("Test Accuracy:", round(rf_test_accuracy, 4))
print("Test Error:", round(rf_test_error, 4))

## Classification Report

In [ ]:
rf_report = classification_report(
    y_test,
    y_pred_rf,
    output_dict=True,
)

rf_report_df = pd.DataFrame(rf_report).transpose()
rf_report_df

In [ ]:
rf_report_df.to_csv(
    TABLE_DIR / "rf_tuned_classification_report.csv",
    index=True,
)

print("Gespeichert:", TABLE_DIR / "rf_tuned_classification_report.csv")

## Confusion Matrix

Die Confusion Matrix zeigt, welche echten Klassen welchen vorhergesagten Klassen zugeordnet wurden.

In [ ]:
labels = sorted(y_train.unique())

rf_cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=labels,
)

rf_cm_table = pd.DataFrame(
    rf_cm,
    index=[f"True: {label}" for label in labels],
    columns=[f"Pred: {label}" for label in labels],
)

rf_cm_table

In [ ]:
rf_cm_table.to_csv(
    TABLE_DIR / "rf_tuned_confusion_matrix.csv",
    index=True,
)

print("Gespeichert:", TABLE_DIR / "rf_tuned_confusion_matrix.csv")

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=labels,
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d",
)

plt.title("Confusion Matrix für getunten RandomForestClassifier")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "rf_tuned_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Testmetriken speichern

In [ ]:
rf_test_metrics = pd.DataFrame({
    "model": ["RandomForestClassifier_tuned"],
    "test_accuracy": [rf_test_accuracy],
    "test_error": [rf_test_error],
    "n_test_observations": [len(y_test)],
})

rf_test_metrics.to_csv(
    TABLE_DIR / "rf_tuned_test_metrics.csv",
    index=False,
)

rf_test_metrics